In [3]:
import torch
import transformers
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import (
    BertTokenizerFast, 
    BertForMaskedLM, 
    DataCollatorForLanguageModeling
)

from optimizers import Alg1Optim

ModuleNotFoundError: No module named 'transformers'

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")

PyTorch version: 2.11.0
Transformers version: 5.5.4


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps") 
else:
    device = torch.device("cpu")  

print(f"Using device: {device}")

model_checkpoint = "bert-base-uncased"
batch_size = 8  
max_length = 128 
epochs = 1


Using device: mps


In [ ]:
tokenizer = BertTokenizerFast.from_pretrained(model_checkpoint)
model = BertForMaskedLM.from_pretrained(model_checkpoint)
model.to(device)

datasets = load_dataset("wikitext", "wikitext-2-raw-v1")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        return_special_tokens_mask=True,
        truncation=True, 
        max_length=max_length
    )

filtered_datasets = datasets.filter(lambda x: len(x["text"]) > 10)

tokenized_datasets = filtered_datasets.map(
    tokenize_function, 
    batched=True, 
    num_proc=4, 
    remove_columns=["text"]
)

Map (num_proc=4):   0%|          | 0/2880 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/23649 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/2461 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=True, 
    mlm_probability=0.15 #mlm - masked language modeling - enables the creation of masked tokens for training
)

train_dataloader = DataLoader(
    tokenized_datasets["train"], 
    shuffle=True, 
    batch_size=batch_size, 
    collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], 
    shuffle=False, 
    batch_size=batch_size, 
    collate_fn=data_collator
)

print(f"Training set batches: {len(train_dataloader)}")
print(f"Validation set batches: {len(val_dataloader)}")

Training set batches: 2957
Validation set batches: 308


In [ ]:
optimizer = Alg1Optim(model.parameters(), lr=1e-4)

In [26]:
from tqdm.auto import tqdm

num_training_steps = epochs * len(train_dataloader)
progress_bar = tqdm(range(num_training_steps))

model.train()
train_losses = []
val_losses = []


for epoch in range(epochs):
    print(f"\n--- Epoch {epoch + 1}/{epochs} ---")
    model.train()
    total_train_loss = 0
    
    for step, batch in enumerate(train_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        train_losses.append(loss.item())
        
        if step % 50 == 0:
            print(f"Step {step} | Loss: {loss.item():.4f}")

        progress_bar.update(1)

    avg_train_loss = total_train_loss / len(train_dataloader)

    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in val_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            total_val_loss += loss.item()
            val_losses.append(loss.item())

    avg_val_loss = total_val_loss / len(val_dataloader)
    val_losses.append(avg_val_loss)

    print(f"Summary of epoch {epoch + 1}: Avg Train Loss: {avg_train_loss:.4f} | Avg Val Loss: {avg_val_loss:.4f}")

  0%|          | 0/2957 [00:00<?, ?it/s]


--- Epoch 1/1 ---
Step 0 | Loss: 2.7201


KeyboardInterrupt: 

In [2]:
period=67